# Library 06 – Figures

The network pictures follow the **same rules in every format**, so they can be compared side by side:

* **Layout:** ForceAtlas2 (NetworkX version) on the unweighted ties, fixed seed. Positions are only a drawing aid.
* **Node size:** strength (total partnership runs).
* **Edge width:** partnership runs of the pair.
* **Node colour:** Louvain community. Communities with fewer than 3 players are grey.
* **Labels:** only the most prominent batters, so the picture stays readable.

Node and edge sizes use fixed reference values, so a node of the same size means the same number of runs in
every panel.

Needs: `lib_00_settings`.

In [ ]:
import matplotlib

matplotlib.use("Agg")   # draw to files, no window needed
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

COMMUNITY_COLOURS = ["#2a78d6", "#eb6834", "#1baf7a", "#6250d6"]   # blue, orange, green, violet
SMALL_COMMUNITY_COLOUR = "#9a9993"
TEXT_COLOUR, MUTED_TEXT_COLOUR, BACKGROUND_COLOUR = "#0b0b0b", "#52514e", "#ffffff"
EDGE_COLOUR = "#8a8a86"
SMALLEST_COLOURED_COMMUNITY = 3

# Fixed scales: the largest strength (15,303) and largest edge weight (2,697) seen in any format.
STRENGTH_REFERENCE = 15303.0
EDGE_WEIGHT_REFERENCE = 2697.0

FIGURE_NOTE = ("Node area: strength (total partnership runs). Edge width: partnership runs of the pair. "
               "Colour: Louvain community. Layout: ForceAtlas2 (positions are not a measure).")


def node_area(strength: float, size_scale: float = 1.0) -> float:
    """Marker area for a node with the given strength."""
    return size_scale * (25 + 1100 * strength / STRENGTH_REFERENCE)


def layout_positions(partnership_graph: nx.Graph) -> dict:
    """Seeded ForceAtlas2 positions computed on the unweighted ties.

    Using run totals as attraction would pull the heavy pairs into one point, so weights are shown through
    edge width instead.
    """
    return nx.forceatlas2_layout(partnership_graph, weight=None, seed=RANDOM_SEED, max_iter=2000,
                                 scaling_ratio=2.0, gravity=1.0)


def community_colours(node_measure_table: pd.DataFrame):
    """Colour for each node, plus the legend entries. Community 1 is the largest."""
    community_sizes = node_measure_table.groupby("community").size()
    colour_of_community, legend_entries, colours_used = {}, [], 0
    for community_number in sorted(community_sizes.index):
        size = community_sizes[community_number]
        if size >= SMALLEST_COLOURED_COMMUNITY and colours_used < len(COMMUNITY_COLOURS):
            colour_of_community[community_number] = COMMUNITY_COLOURS[colours_used]
            legend_entries.append((f"Community {community_number} ({size} players)", COMMUNITY_COLOURS[colours_used]))
            colours_used += 1
        else:
            colour_of_community[community_number] = SMALL_COMMUNITY_COLOUR
    if SMALL_COMMUNITY_COLOUR in colour_of_community.values():
        legend_entries.append((f"Communities of < {SMALLEST_COLOURED_COMMUNITY} players", SMALL_COMMUNITY_COLOUR))
    colour_of_node = {row.player_id: colour_of_community[row.community] for row in node_measure_table.itertuples()}
    return colour_of_node, legend_entries

## Drawing one network

In [ ]:
def draw_network(axis, partnership_graph: nx.Graph, node_measure_table: pd.DataFrame, title: str,
                 labelled_batters: int = 14, size_scale: float = 1.0, font_size: float = 8.0) -> None:
    """Draw one format network on ``axis`` using the shared rules above."""
    positions = layout_positions(partnership_graph)
    colour_of_node, legend_entries = community_colours(node_measure_table)
    strength_of = node_measure_table.set_index("player_id")["strength"]
    node_list = list(partnership_graph.nodes)
    node_areas = [node_area(strength_of[player], size_scale) for player in node_list]
    edge_weights = np.array([attributes["weight"] for _, _, attributes in partnership_graph.edges(data=True)], dtype=float)
    edge_widths = 0.25 + 5.0 * edge_weights / EDGE_WEIGHT_REFERENCE

    nx.draw_networkx_edges(partnership_graph, positions, ax=axis, width=edge_widths, edge_color=EDGE_COLOUR, alpha=0.45)
    nx.draw_networkx_nodes(partnership_graph, positions, ax=axis, node_size=node_areas,
                           node_color=[colour_of_node[player] for player in node_list],
                           edgecolors=BACKGROUND_COLOUR, linewidths=0.9)

    # Label the batters with the highest strength, plus the four with the highest betweenness.
    batters_to_label = (set(node_measure_table.nlargest(labelled_batters, "strength")["player_id"])
                        | set(node_measure_table.nlargest(4, "betweenness")["player_id"]))
    for player_id in sorted(batters_to_label):   # fixed order, so overlapping labels always stack the same way
        x_position, y_position = positions[player_id]
        label_offset = 6 + np.sqrt(node_areas[node_list.index(player_id)]) / 2.4
        axis.annotate(partnership_graph.nodes[player_id]["player_name"], (x_position, y_position),
                      xytext=(0, label_offset), textcoords="offset points", ha="center",
                      fontsize=font_size, color=TEXT_COLOUR,
                      bbox=dict(boxstyle="round,pad=0.12", fc=BACKGROUND_COLOUR, ec="none", alpha=0.75))

    axis.set_title(title, fontsize=font_size + 3, color=TEXT_COLOUR, loc="left")
    legend_handles = [Line2D([], [], marker="o", ls="", ms=7, mfc=colour, mec=BACKGROUND_COLOUR, label=label)
                      for label, colour in legend_entries]
    axis.legend(handles=legend_handles, loc="lower left", fontsize=font_size - 1, frameon=False,
                labelcolor=MUTED_TEXT_COLOUR)
    axis.set_axis_off()
    axis.margins(0.08)

## The three report figures

In [ ]:
def save_single_network_figure(partnership_graph: nx.Graph, node_measure_table: pd.DataFrame,
                               format_name: str, output_path: Path) -> None:
    """One large figure for one format."""
    figure, axis = plt.subplots(figsize=(9, 8), facecolor=BACKGROUND_COLOUR)
    draw_network(axis, partnership_graph, node_measure_table,
                 f"England Men {format_name} batting-partnership network "
                 f"({partnership_graph.number_of_nodes()} batters, {partnership_graph.number_of_edges()} pairs)",
                 size_scale=1.6, font_size=9)
    figure.text(0.02, 0.012, FIGURE_NOTE, fontsize=7.5, color=MUTED_TEXT_COLOUR)
    figure.savefig(output_path, dpi=300, bbox_inches="tight", facecolor=BACKGROUND_COLOUR)
    plt.close(figure)


def save_comparison_figure(graphs_by_format: dict, measures_by_format: dict, output_path: Path) -> None:
    """2 x 2 grid: one panel per format and one panel with the size key."""
    figure, axes = plt.subplots(2, 2, figsize=(9.4, 9.6), facecolor=BACKGROUND_COLOUR)
    panels = axes.ravel()
    for axis, format_name in zip(panels, FORMATS):
        partnership_graph = graphs_by_format[format_name]
        draw_network(axis, partnership_graph, measures_by_format[format_name],
                     f"{format_name}: {partnership_graph.number_of_nodes()} batters, "
                     f"{partnership_graph.number_of_edges()} pairs",
                     labelled_batters=9, size_scale=0.75, font_size=9)

    key_panel = panels[3]
    key_panel.set_axis_off()
    key_panel.set_xlim(0, 1)
    key_panel.set_ylim(0, 1)
    key_panel.text(0.05, 0.93, "Key (same scales in all panels)", fontsize=11, color=TEXT_COLOUR)
    for row_number, example_runs in enumerate((500, 5000, 15000)):
        y_position = 0.80 - 0.11 * row_number
        key_panel.scatter([0.12], [y_position], s=node_area(example_runs, 0.75), color=SMALL_COMMUNITY_COLOUR,
                          edgecolors=BACKGROUND_COLOUR)
        key_panel.text(0.24, y_position, f"strength {example_runs:,} runs", va="center", fontsize=9, color=MUTED_TEXT_COLOUR)
    for row_number, example_runs in enumerate((50, 500, 2500)):
        y_position = 0.44 - 0.07 * row_number
        key_panel.plot([0.05, 0.19], [y_position, y_position], lw=0.25 + 5.0 * example_runs / EDGE_WEIGHT_REFERENCE,
                       color=EDGE_COLOUR, solid_capstyle="round")
        key_panel.text(0.24, y_position, f"pair total {example_runs:,} runs", va="center", fontsize=9,
                       color=MUTED_TEXT_COLOUR)
    key_panel.text(0.05, 0.16, "Colour: Louvain community, numbered by size\nwithin each format.\n"
                   "Layout: ForceAtlas2 on the unweighted ties;\npositions are a drawing aid, not a measure.",
                   fontsize=9, color=MUTED_TEXT_COLOUR, va="center", linespacing=1.35)
    figure.subplots_adjust(wspace=0.02, hspace=0.06, left=0.01, right=0.99, top=0.97, bottom=0.01)
    figure.savefig(output_path, dpi=300, bbox_inches="tight", facecolor=BACKGROUND_COLOUR)
    plt.close(figure)


def save_degree_distribution_figure(measures_by_format: dict, output_path: Path) -> None:
    """Small multiples: how many batters have each number of partners, per format (shared axes)."""
    figure, axes = plt.subplots(1, 3, figsize=(10, 2.9), sharex=True, sharey=True, facecolor=BACKGROUND_COLOUR)
    histogram_bins = np.arange(0, 42, 3)
    for axis, format_name in zip(axes, FORMATS):
        degrees = measures_by_format[format_name]["degree"]
        axis.hist(degrees, bins=histogram_bins, color=COMMUNITY_COLOURS[0], edgecolor=BACKGROUND_COLOUR, linewidth=1.2)
        axis.axvline(degrees.mean(), color=TEXT_COLOUR, lw=1, ls="--")
        axis.text(degrees.mean() + 0.8, 0.93, f"mean {degrees.mean():.1f}", transform=axis.get_xaxis_transform(),
                  fontsize=8, color=MUTED_TEXT_COLOUR)
        axis.set_title(format_name, fontsize=10, loc="left", color=TEXT_COLOUR)
        axis.set_xlabel("Degree (distinct partners)", fontsize=8.5, color=MUTED_TEXT_COLOUR)
        for hidden_side in ("top", "right"):
            axis.spines[hidden_side].set_visible(False)
        axis.spines["left"].set_color("#c9c8c2")
        axis.spines["bottom"].set_color("#c9c8c2")
        axis.tick_params(labelsize=8, colors=MUTED_TEXT_COLOUR, length=0)
        axis.grid(axis="y", color="#ecebe6", lw=0.8)
        axis.set_axisbelow(True)
    axes[0].set_ylabel("Number of batters", fontsize=8.5, color=MUTED_TEXT_COLOUR)
    figure.tight_layout()
    figure.savefig(output_path, dpi=300, bbox_inches="tight", facecolor=BACKGROUND_COLOUR)
    plt.close(figure)